# LeetCode 1333 — Filter Restaurants by Vegan-Friendly, Price and Distance bằng Python

Link đề: https://leetcode.com/problems/filter-restaurants-by-vegan-friendly-price-and-distance/ · Độ khó: **Medium** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def filterRestaurants(self, restaurants: list[list[int]], veganFriendly: int, maxPrice: int, maxDistance: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ đề **lọc + sắp xếp**: hiểu đúng ý nghĩa bộ lọc `veganFriendly = 0` ("không lọc"), không phải "chỉ lấy quán không chay".
2. Thành thạo `sorted`/`list.sort` với **khoá tuple** và mẹo **đổi dấu** để sắp giảm dần nhiều tiêu chí.
3. Hiểu **tính ổn định (stability)** của sắp xếp trong Python và kỹ thuật **sắp nhiều lượt** (tiêu chí phụ trước, tiêu chí chính sau).
4. Biết viết **bộ so sánh** bằng `functools.cmp_to_key` và một thuật toán sắp xếp tự viết (baseline) để hiểu bản chất.
5. Tránh lỗi: lọc `vegan == veganFriendly`, quên tiêu chí phụ `id`, dùng `<` thay `<=`, trả về cả hàng thay vì `id`, sắp nhiều lượt sai thứ tự.

## 2. Đề bài & ví dụ

Cho mảng `restaurants`, mỗi phần tử `[id, rating, veganFriendly, price, distance]`. Lọc bằng ba bộ lọc:

- `veganFriendly`: nếu là `1` (true) thì **chỉ giữ** quán có `veganFriendly_i = 1`; nếu là `0` (false) thì **giữ mọi quán**.
- `maxPrice`, `maxDistance`: chỉ giữ quán có `price_i <= maxPrice` và `distance_i <= maxDistance`.

Trả về danh sách **id** các quán sau khi lọc, sắp theo **rating giảm dần**; nếu cùng rating thì theo **id giảm dần**.

**Ví dụ 1:** `restaurants = [[1,4,1,40,10],[2,8,0,50,5],[3,8,1,30,4],[4,10,0,10,3],[5,1,1,15,1]]`, `veganFriendly = 1, maxPrice = 50, maxDistance = 10` → `[3,1,5]`

**Ví dụ 2:** cùng mảng, `veganFriendly = 0, maxPrice = 50, maxDistance = 10` → `[4,3,2,1,5]`

**Ví dụ 3:** cùng mảng, `veganFriendly = 0, maxPrice = 30, maxDistance = 3` → `[4,5]`

**Ràng buộc:** `1 <= restaurants.length <= 10^4`; `1 <= id, rating, price, distance <= 10^5`; `1 <= maxPrice, maxDistance <= 10^5`; `veganFriendly_i` và `veganFriendly` là `0` hoặc `1`; mọi `id` phân biệt.

In [1]:
import heapq
import random
import time
from functools import cmp_to_key
from typing import List

R = [[1, 4, 1, 40, 10], [2, 8, 0, 50, 5], [3, 8, 1, 30, 4], [4, 10, 0, 10, 3], [5, 1, 1, 15, 1]]
examples = [
    (R, 1, 50, 10, [3, 1, 5]),
    (R, 0, 50, 10, [4, 3, 2, 1, 5]),
    (R, 0, 30, 3, [4, 5]),
]
print("id rating vegan price dist")
for r in R:
    print(" ".join(f"{x:5}" for x in r))

id rating vegan price dist
    1     4     1    40    10
    2     8     0    50     5
    3     8     1    30     4
    4    10     0    10     3
    5     1     1    15     1


## 3. Kiến thức nền

### 3.1 Giải nén (unpacking) một hàng

`id_, rating, vegan, price, dist = r` gán 5 biến trong một dòng — code dễ đọc hơn nhiều so với `r[0]`, `r[3]`… (Tránh đặt tên biến `id` vì che mất hàm có sẵn `id()`.)

In [2]:
r = [3, 8, 1, 30, 4]
id_, rating, vegan, price, dist = r
print(f"id={id_}, rating={rating}, vegan={vegan}, price={price}, dist={dist}")

id=3, rating=8, vegan=1, price=30, dist=4


### 3.2 List comprehension có điều kiện

`[f(x) for x in xs if cond(x)]` vừa lọc vừa biến đổi. Điều kiện ghép bằng `and`/`or`.

In [3]:
nums = [5, 12, 7, 20, 3]
print([x * 10 for x in nums if x > 4 and x % 2 == 1])

[50, 70]


### 3.3 Sắp xếp với `key` là tuple

Python so sánh tuple **từng phần tử từ trái sang** (thứ tự từ điển). `key=lambda r: (a, b)` ⇒ sắp theo `a`, hoà thì theo `b`. Muốn **giảm dần** một tiêu chí số ⇒ đổi dấu `-a`.

In [4]:
pairs = [(8, 2), (10, 4), (8, 3), (4, 1)]
print("tăng theo (a, b):          ", sorted(pairs))
print("giảm a, giảm b (đổi dấu):   ", sorted(pairs, key=lambda p: (-p[0], -p[1])))
print("giảm a, giảm b (reverse):   ", sorted(pairs, reverse=True))
print("giảm a, TĂNG b:             ", sorted(pairs, key=lambda p: (-p[0], p[1])))

tăng theo (a, b):           [(4, 1), (8, 2), (8, 3), (10, 4)]
giảm a, giảm b (đổi dấu):    [(10, 4), (8, 3), (8, 2), (4, 1)]
giảm a, giảm b (reverse):    [(10, 4), (8, 3), (8, 2), (4, 1)]
giảm a, TĂNG b:              [(10, 4), (8, 2), (8, 3), (4, 1)]


### 3.4 Tính ổn định của sắp xếp

Sắp xếp của Python (Timsort) là **ổn định**: các phần tử có khoá bằng nhau giữ nguyên thứ tự tương đối ban đầu. Nhờ vậy có thể sắp **nhiều lượt**: sắp theo tiêu chí **phụ** trước, rồi theo tiêu chí **chính** — kết quả tương đương sắp theo `(chính, phụ)`.

In [5]:
people = [("An", 20), ("Binh", 18), ("Chi", 20), ("Dung", 18)]
step1 = sorted(people, key=lambda p: p[0], reverse=True)    # phụ: tên giảm dần
step2 = sorted(step1, key=lambda p: p[1], reverse=True)     # chính: tuổi giảm dần
print(step1)
print(step2)

[('Dung', 18), ('Chi', 20), ('Binh', 18), ('An', 20)]
[('Chi', 20), ('An', 20), ('Dung', 18), ('Binh', 18)]


### 3.5 Bộ so sánh `cmp_to_key`

Đôi khi dễ diễn đạt bằng hàm so sánh `cmp(a, b)` trả số âm (a đứng trước), `0`, hoặc dương (b đứng trước). `functools.cmp_to_key(cmp)` chuyển nó thành `key`.

In [6]:
def cmp(a, b):
    if a[0] != b[0]:
        return b[0] - a[0]          # a lớn hơn thì đứng trước
    return b[1] - a[1]

print(sorted(pairs, key=cmp_to_key(cmp)))

[(10, 4), (8, 3), (8, 2), (4, 1)]


### 3.6 Logic của bộ lọc chay: `not veganFriendly or vegan == 1`

"Nếu yêu cầu chay thì quán phải chay; không yêu cầu thì quán nào cũng được" ⇒ điều kiện giữ: `veganFriendly == 0 or vegan_i == 1`. Vì cả hai là 0/1, có thể viết gọn `vegan_i >= veganFriendly`.

In [7]:
for want in (0, 1):
    for have in (0, 1):
        print(f"veganFriendly={want}, quán vegan={have}: giữ? {want == 0 or have == 1} | gọn: {have >= want}")

veganFriendly=0, quán vegan=0: giữ? True | gọn: True
veganFriendly=0, quán vegan=1: giữ? True | gọn: True
veganFriendly=1, quán vegan=0: giữ? False | gọn: False
veganFriendly=1, quán vegan=1: giữ? True | gọn: True


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (`veganFriendly = 1, maxPrice = 50, maxDistance = 10`):

| id | rating | vegan | price ≤ 50? | dist ≤ 10? | vegan ok? | giữ? |
|---|---|---|---|---|---|---|
| 1 | 4 | 1 | 40 ✔ | 10 ✔ | ✔ | ✔ |
| 2 | 8 | 0 | 50 ✔ | 5 ✔ | ✗ | ✗ |
| 3 | 8 | 1 | 30 ✔ | 4 ✔ | ✔ | ✔ |
| 4 | 10 | 0 | 10 ✔ | 3 ✔ | ✗ | ✗ |
| 5 | 1 | 1 | 15 ✔ | 1 ✔ | ✔ | ✔ |

Còn lại `(id, rating)`: `(1,4), (3,8), (5,1)`. Sắp rating giảm dần: `3 (8), 1 (4), 5 (1)` ⇒ `[3,1,5]` ✔.

**Ví dụ 2** (không lọc chay): giữ cả 5 quán. Rating: `4→10, 2→8, 3→8, 1→4, 5→1`. Hai quán rating 8 là `2` và `3` ⇒ id giảm dần ⇒ `3` trước `2` ⇒ `[4,3,2,1,5]` ✔. Đây là chỗ tiêu chí phụ phát huy tác dụng.

In [8]:
for rest, vf, mp, md_, want in examples:
    kept = [(r[0], r[1]) for r in rest if (vf == 0 or r[2] == 1) and r[3] <= mp and r[4] <= md_]
    print(f"vf={vf}, maxPrice={mp}, maxDistance={md_}: giữ (id, rating) = {kept} -> mong đợi {want}")

vf=1, maxPrice=50, maxDistance=10: giữ (id, rating) = [(1, 4), (3, 8), (5, 1)] -> mong đợi [3, 1, 5]
vf=0, maxPrice=50, maxDistance=10: giữ (id, rating) = [(1, 4), (2, 8), (3, 8), (4, 10), (5, 1)] -> mong đợi [4, 3, 2, 1, 5]
vf=0, maxPrice=30, maxDistance=3: giữ (id, rating) = [(4, 10), (5, 1)] -> mong đợi [4, 5]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lọc bằng vòng lặp `for` thường, rồi sắp bằng **selection sort tự viết**: mỗi bước tìm quán "tốt nhất" còn lại (rating lớn nhất, hoà thì id lớn nhất), đưa vào kết quả. Cách này không dùng thư viện sắp xếp — giúp thấy rõ phép so sánh "đứng trước".

In [9]:
def better(a, b):
    """a đứng trước b? (a, b là [id, rating, ...])"""
    if a[1] != b[1]:
        return a[1] > b[1]
    return a[0] > b[0]

def filter_restaurants_brute(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = []
    for r in restaurants:
        id_, rating, vegan, price, dist = r
        if veganFriendly == 1 and vegan == 0:
            continue
        if price > maxPrice or dist > maxDistance:
            continue
        kept.append(r)
    res = []
    while kept:
        best_i = 0
        for i in range(1, len(kept)):
            if better(kept[i], kept[best_i]):
                best_i = i
        res.append(kept.pop(best_i)[0])
    return res

for rest, vf, mp, md_, want in examples:
    got = filter_restaurants_brute(rest, vf, mp, md_)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[3, 1, 5] | mong đợi [3, 1, 5] | OK
[4, 3, 2, 1, 5] | mong đợi [4, 3, 2, 1, 5] | OK
[4, 5] | mong đợi [4, 5] | OK


**Độ phức tạp:** lọc `O(n)`; selection sort `O(k²)` với `k` = số quán còn lại (xấu nhất `k = n = 10^4` ⇒ `10^8` phép so sánh ở Python — quá chậm). Bộ nhớ `O(k)`.

**Nút thắt:** tự sắp xếp bằng `O(k²)`. Python có sẵn Timsort `O(k log k)`; ta chỉ cần biểu diễn đúng quy tắc "đứng trước" dưới dạng **khoá**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** quy tắc "rating giảm dần, hoà thì id giảm dần" chính là so sánh tuple `(rating, id)` theo chiều **giảm**. Vì mọi `id` phân biệt, thứ tự này là **toàn phần** (không có hai quán "hoà hoàn toàn") ⇒ đáp án duy nhất.

Ba cách biểu diễn:
1. `key=lambda r: (-r[1], -r[0])` — đổi dấu, sắp tăng.
2. `key=lambda r: (r[1], r[0]), reverse=True` — sắp giảm cả tuple.
3. Hai lượt ổn định: theo `id` giảm, rồi theo `rating` giảm.

**Quy tắc lọc "nếu… thì…":**
- Nếu `veganFriendly == 1` và quán không chay ⇒ loại.
- Nếu `price > maxPrice` hoặc `distance > maxDistance` ⇒ loại.
- Còn lại ⇒ giữ.

**Thứ tự thao tác:** lọc **trước** rồi mới sắp ⇒ sắp ít phần tử hơn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Lọc bằng comprehension + `sorted` với khoá đổi dấu

**Từng đoạn:**
1. `kept = [r for r in restaurants if (vf == 0 or r[2] == 1) and r[3] <= mp and r[4] <= md]`.
2. `kept.sort(key=lambda r: (-r[1], -r[0]))`.
3. `return [r[0] for r in kept]`.

In [10]:
def filter_restaurants_key(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants
            if (veganFriendly == 0 or r[2] == 1) and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=lambda r: (-r[1], -r[0]))
    return [r[0] for r in kept]

for rest, vf, mp, md_, want in examples:
    got = filter_restaurants_key(rest, vf, mp, md_)
    print(got, "| OK" if got == want else "| SAI")

[3, 1, 5] | OK
[4, 3, 2, 1, 5] | OK
[4, 5] | OK


Code nộp LeetCode (Cách 2):

In [11]:
class SolutionKey:
    def filterRestaurants(self, restaurants: list[list[int]], veganFriendly: int, maxPrice: int, maxDistance: int) -> list[int]:
        kept = [r for r in restaurants
                if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
        kept.sort(key=lambda r: (-r[1], -r[0]))
        return [r[0] for r in kept]

print([SolutionKey().filterRestaurants(*e[:4]) for e in examples])

[[3, 1, 5], [4, 3, 2, 1, 5], [4, 5]]


**Độ phức tạp:** `O(n + k log k)` thời gian, `O(k)` bộ nhớ.

### 7.2 Cách 3 — Chỉ giữ cặp `(rating, id)` và `sort(reverse=True)`

Ta không cần giữ cả hàng 5 phần tử: chỉ cần cặp `(rating, id)`. Sắp giảm dần **tự nhiên** theo tuple, rồi lấy `id`. Khoá tuple tự nhiên còn nhanh hơn lambda (so sánh tuple trực tiếp, không gọi hàm Python cho mỗi phần tử).

In [12]:
def filter_restaurants_pairs(restaurants, veganFriendly, maxPrice, maxDistance):
    pairs = [(rating, id_) for id_, rating, vegan, price, dist in restaurants
             if vegan >= veganFriendly and price <= maxPrice and dist <= maxDistance]
    pairs.sort(reverse=True)
    return [id_ for _, id_ in pairs]

for rest, vf, mp, md_, want in examples:
    got = filter_restaurants_pairs(rest, vf, mp, md_)
    print(got, "| OK" if got == want else "| SAI")

[3, 1, 5] | OK
[4, 3, 2, 1, 5] | OK
[4, 5] | OK


Code nộp LeetCode (Cách 3):

In [13]:
class SolutionPairs:
    def filterRestaurants(self, restaurants: list[list[int]], veganFriendly: int, maxPrice: int, maxDistance: int) -> list[int]:
        pairs = sorted(((rating, id_) for id_, rating, vegan, price, dist in restaurants
                        if vegan >= veganFriendly and price <= maxPrice and dist <= maxDistance),
                       reverse=True)
        return [id_ for _, id_ in pairs]

print([SolutionPairs().filterRestaurants(*e[:4]) for e in examples])

[[3, 1, 5], [4, 3, 2, 1, 5], [4, 5]]


**Độ phức tạp:** `O(n + k log k)` thời gian, `O(k)` bộ nhớ.

### 7.3 Cách 4 — Sắp hai lượt nhờ tính ổn định

**Ý tưởng:** sắp theo tiêu chí **phụ** (`id` giảm dần) trước, sau đó sắp theo tiêu chí **chính** (`rating` giảm dần). Vì Timsort ổn định, các quán cùng rating vẫn giữ thứ tự `id` giảm dần từ lượt 1. Kỹ thuật này đặc biệt hữu ích khi các tiêu chí có chiều khác nhau và **không đổi dấu được** (ví dụ chuỗi: giảm dần theo tên, tăng dần theo tuổi).

Lưu ý: `reverse=True` trong Python **vẫn giữ ổn định** (các phần tử bằng nhau không bị đảo thứ tự).

In [14]:
def filter_restaurants_two_pass(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants
            if (veganFriendly == 0 or r[2] == 1) and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=lambda r: r[0], reverse=True)     # lượt 1: phụ — id giảm
    kept.sort(key=lambda r: r[1], reverse=True)     # lượt 2: chính — rating giảm (ổn định)
    return [r[0] for r in kept]

for rest, vf, mp, md_, want in examples:
    got = filter_restaurants_two_pass(rest, vf, mp, md_)
    print(got, "| OK" if got == want else "| SAI")

[3, 1, 5] | OK
[4, 3, 2, 1, 5] | OK
[4, 5] | OK


Code nộp LeetCode (Cách 4):

In [15]:
class SolutionTwoPass:
    def filterRestaurants(self, restaurants: list[list[int]], veganFriendly: int, maxPrice: int, maxDistance: int) -> list[int]:
        kept = [r for r in restaurants
                if (veganFriendly == 0 or r[2] == 1) and r[3] <= maxPrice and r[4] <= maxDistance]
        kept.sort(key=lambda r: r[0], reverse=True)
        kept.sort(key=lambda r: r[1], reverse=True)
        return [r[0] for r in kept]

print([SolutionTwoPass().filterRestaurants(*e[:4]) for e in examples])

[[3, 1, 5], [4, 3, 2, 1, 5], [4, 5]]


**Độ phức tạp:** hai lần sắp ⇒ vẫn `O(n + k log k)`, hằng số gấp đôi.

### 7.4 Cách 5 — Bộ so sánh với `cmp_to_key`

Viết quy tắc "đứng trước" như hàm so sánh — gần với cách viết `Comparator` trong Java/C++. Chậm hơn khoá tuple (gọi hàm Python cho mỗi phép so sánh) nhưng linh hoạt nhất.

In [16]:
def filter_restaurants_cmp(restaurants, veganFriendly, maxPrice, maxDistance):
    def cmp(a, b):
        if a[1] != b[1]:
            return -1 if a[1] > b[1] else 1
        return -1 if a[0] > b[0] else (1 if a[0] < b[0] else 0)

    kept = [r for r in restaurants
            if (veganFriendly == 0 or r[2] == 1) and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=cmp_to_key(cmp))
    return [r[0] for r in kept]

for rest, vf, mp, md_, want in examples:
    print(filter_restaurants_cmp(rest, vf, mp, md_), "| mong đợi", want)

[3, 1, 5] | mong đợi [3, 1, 5]
[4, 3, 2, 1, 5] | mong đợi [4, 3, 2, 1, 5]
[4, 5] | mong đợi [4, 5]


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** (`veganFriendly = 0, maxPrice = 50, maxDistance = 10`):

| r | vegan ≥ 0? | price ≤ 50? | dist ≤ 10? | giữ | pairs sau bước |
|---|---|---|---|---|---|
| [1,4,1,40,10] | ✔ | ✔ | ✔ | ✔ | (4,1) |
| [2,8,0,50,5] | ✔ | ✔ | ✔ | ✔ | (4,1),(8,2) |
| [3,8,1,30,4] | ✔ | ✔ | ✔ | ✔ | …,(8,3) |
| [4,10,0,10,3] | ✔ | ✔ | ✔ | ✔ | …,(10,4) |
| [5,1,1,15,1] | ✔ | ✔ | ✔ | ✔ | …,(1,5) |

Sắp giảm dần: `(10,4), (8,3), (8,2), (4,1), (1,5)` ⇒ `[4,3,2,1,5]`.

In [17]:
def trace_filter(restaurants, veganFriendly, maxPrice, maxDistance):
    pairs = []
    for id_, rating, vegan, price, dist in restaurants:
        ok_v = vegan >= veganFriendly
        ok_p = price <= maxPrice
        ok_d = dist <= maxDistance
        keep = ok_v and ok_p and ok_d
        if keep:
            pairs.append((rating, id_))
        print(f"id={id_}: vegan_ok={ok_v!s:5} price_ok={ok_p!s:5} dist_ok={ok_d!s:5} -> {'giữ' if keep else 'loại'} | pairs={pairs}")
    pairs.sort(reverse=True)
    print("sau khi sắp giảm dần:", pairs)
    res = [i for _, i in pairs]
    print("kết quả:", res)
    return res

trace_filter(R, 0, 50, 10)

id=1: vegan_ok=True  price_ok=True  dist_ok=True  -> giữ | pairs=[(4, 1)]
id=2: vegan_ok=True  price_ok=True  dist_ok=True  -> giữ | pairs=[(4, 1), (8, 2)]
id=3: vegan_ok=True  price_ok=True  dist_ok=True  -> giữ | pairs=[(4, 1), (8, 2), (8, 3)]
id=4: vegan_ok=True  price_ok=True  dist_ok=True  -> giữ | pairs=[(4, 1), (8, 2), (8, 3), (10, 4)]
id=5: vegan_ok=True  price_ok=True  dist_ok=True  -> giữ | pairs=[(4, 1), (8, 2), (8, 3), (10, 4), (1, 5)]
sau khi sắp giảm dần: [(10, 4), (8, 3), (8, 2), (4, 1), (1, 5)]
kết quả: [4, 3, 2, 1, 5]


[4, 3, 2, 1, 5]

Trace Cách 4 (hai lượt) — xem thứ tự sau từng lượt:

In [18]:
def trace_two_pass(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
    print("sau lọc        :", [(r[0], r[1]) for r in kept], "(id, rating)")
    kept.sort(key=lambda r: r[0], reverse=True)
    print("lượt 1 (id↓)   :", [(r[0], r[1]) for r in kept])
    kept.sort(key=lambda r: r[1], reverse=True)
    print("lượt 2 (rating↓):", [(r[0], r[1]) for r in kept])

trace_two_pass(R, 0, 50, 10)

sau lọc        : [(1, 4), (2, 8), (3, 8), (4, 10), (5, 1)] (id, rating)
lượt 1 (id↓)   : [(5, 1), (4, 10), (3, 8), (2, 8), (1, 4)]
lượt 2 (rating↓): [(4, 10), (3, 8), (2, 8), (1, 4), (5, 1)]


## 9. Code nộp LeetCode

In [19]:
class Solution:
    def filterRestaurants(self, restaurants: list[list[int]], veganFriendly: int, maxPrice: int, maxDistance: int) -> list[int]:
        kept = [(rating, id_) for id_, rating, vegan, price, dist in restaurants
                if (veganFriendly == 0 or vegan == 1)
                and price <= maxPrice and dist <= maxDistance]
        kept.sort(reverse=True)                      # rating giảm, hoà thì id giảm
        return [id_ for _, id_ in kept]

for rest, vf, mp, md_, want in examples:
    got = Solution().filterRestaurants(rest, vf, mp, md_)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[3, 1, 5] | mong đợi [3, 1, 5] | OK
[4, 3, 2, 1, 5] | mong đợi [4, 3, 2, 1, 5] | OK
[4, 5] | mong đợi [4, 5] | OK


## 10. Vì sao thuật toán đúng

1. **Lọc đúng:** một quán được giữ ⇔ thoả cả ba điều kiện của đề: (a) `veganFriendly == 0` hoặc quán chay; (b) `price <= maxPrice`; (c) `distance <= maxDistance` — comprehension kiểm tra đúng hội ba điều kiện này.
2. **Thứ tự đúng:** với hai quán `A`, `B` còn lại, đề muốn `A` trước `B` ⇔ `rating_A > rating_B` hoặc (`rating_A == rating_B` và `id_A > id_B`) ⇔ `(rating_A, id_A) > (rating_B, id_B)` theo so sánh tuple. Sắp tuple giảm dần thực hiện đúng quan hệ này.
3. **Duy nhất:** `id` phân biệt ⇒ không có hai tuple bằng nhau ⇒ thứ tự toàn phần, đáp án là duy nhất (không phụ thuộc thứ tự đầu vào).
4. **Hai lượt (Cách 4):** sau lượt 1, danh sách sắp theo `id` giảm. Lượt 2 sắp theo `rating` giảm và **ổn định**, nên trong mỗi nhóm cùng rating, thứ tự từ lượt 1 (id giảm) được giữ ⇒ đúng quy tắc.

## 11. Phân tích Time / Space

Đặt `n = len(restaurants)` (`≤ 10^4`), `k` = số quán còn lại sau lọc (`k ≤ n`).

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp + selection sort | `O(n + k²)` | `O(k)` |
| 2. Comprehension + key đổi dấu | `O(n + k log k)` | `O(k)` |
| 3. Cặp `(rating, id)` + reverse | `O(n + k log k)` | `O(k)` |
| 4. Hai lượt ổn định | `O(n + k log k)` | `O(k)` |
| 5. `cmp_to_key` | `O(n + k log k)` (hằng số lớn) | `O(k)` |

Sắp xếp dựa trên so sánh không thể nhanh hơn `O(k log k)` trong trường hợp tổng quát.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
solvers = {
    "brute": filter_restaurants_brute,
    "key": filter_restaurants_key,
    "pairs": filter_restaurants_pairs,
    "two_pass": filter_restaurants_two_pass,
    "cmp": filter_restaurants_cmp,
    "submit": lambda *a: Solution().filterRestaurants(*a),
    "submit_key": lambda *a: SolutionKey().filterRestaurants(*a),
    "submit_pairs": lambda *a: SolutionPairs().filterRestaurants(*a),
    "submit_two": lambda *a: SolutionTwoPass().filterRestaurants(*a),
}
cases = examples + [
    ([[7, 5, 0, 10, 10]], 1, 100, 100, []),                       # yêu cầu chay, quán không chay
    ([[7, 5, 0, 10, 10]], 0, 10, 10, [7]),                        # biên đúng bằng max (<=)
    ([[7, 5, 0, 10, 10]], 0, 9, 10, []),                          # vượt giá 1 đơn vị
    ([[1, 5, 1, 1, 1], [9, 5, 1, 1, 1], [4, 5, 1, 1, 1]], 1, 1, 1, [9, 4, 1]),  # cùng rating -> id giảm
    ([[10, 1, 0, 1, 1], [2, 3, 0, 1, 1]], 0, 5, 5, [2, 10]),      # id không phải chỉ số mảng
]
for rest, vf, mp, md_, want in cases:
    for name, f in solvers.items():
        assert f(rest, vf, mp, md_) == want, (name, rest, vf, mp, md_)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 8 case qua cho 9 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh `rating` trong miền nhỏ để có nhiều hoà; `id` phân biệt và **xáo trộn** (không trùng chỉ số).

In [21]:
def random_case(rng, n, hi=10**5, rating_hi=5):
    ids = rng.sample(range(1, hi + 1), n)
    rest = [[ids[i], rng.randint(1, rating_hi), rng.randint(0, 1), rng.randint(1, 50), rng.randint(1, 50)]
            for i in range(n)]
    return rest, rng.randint(0, 1), rng.randint(1, 50), rng.randint(1, 50)

def is_valid(rest, vf, mp, md_, out):
    ok = {r[0] for r in rest if (vf == 0 or r[2] == 1) and r[3] <= mp and r[4] <= md_}
    if set(out) != ok or len(out) != len(ok):
        return False
    info = {r[0]: r[1] for r in rest}
    return all((info[a], a) > (info[b], b) for a, b in zip(out, out[1:]))

rng = random.Random(1333)
for _ in range(1000):
    args = random_case(rng, rng.randint(1, 30))
    want = filter_restaurants_brute(*args)
    assert is_valid(*args, want)
    for name, f in solvers.items():
        assert f(*args) == want, (name, args)
print("1000 bộ ngẫu nhiên: mọi cách khớp brute force và hợp lệ")

1000 bộ ngẫu nhiên: mọi cách khớp brute force và hợp lệ


### 12.3 Kích thước lớn nhất `n = 10^4`

In [22]:
rng = random.Random(7)
big = random_case(rng, 10**4, rating_hi=10**5)
big = (big[0], 0, 10**5, 10**5)          # không lọc gì -> sắp cả 10^4 quán
for name in ["pairs", "key", "two_pass", "cmp", "submit"]:
    t0 = time.time()
    out = solvers[name](*big)
    print(f"{name:9}: {len(out)} quán, {(time.time() - t0) * 1000:6.1f} ms")
assert is_valid(*big, Solution().filterRestaurants(*big))
small = random_case(random.Random(8), 2000, rating_hi=50)
small = (small[0], 0, 10**5, 10**5)   # không lọc -> k = 2000
t0 = time.time(); filter_restaurants_brute(*small); print(f"brute với n=2000: {(time.time() - t0) * 1000:.0f} ms (O(k^2))")

pairs    : 10000 quán,    3.2 ms


key      : 10000 quán,    6.0 ms
two_pass : 10000 quán,    4.2 ms


cmp      : 10000 quán,   12.4 ms
submit   : 10000 quán,    3.5 ms


brute với n=2000: 149 ms (O(k^2))


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hiểu sai bộ lọc chay: `vegan == veganFriendly`

In [23]:
def wrong_vegan_equal(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [(r[1], r[0]) for r in restaurants
            if r[2] == veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]   # SAI
    kept.sort(reverse=True)
    return [i for _, i in kept]

rest, vf, mp, md_, want = examples[1]
print("sai :", wrong_vegan_equal(rest, vf, mp, md_), "| đúng:", Solution().filterRestaurants(rest, vf, mp, md_))

sai : [4, 2] | đúng: [4, 3, 2, 1, 5]


**Vì sao sai:** `veganFriendly = 0` nghĩa là "**không** áp dụng bộ lọc chay", không phải "chỉ lấy quán không chay". Code sai loại mất quán `1, 3, 5`. **Sửa:** `veganFriendly == 0 or vegan == 1` (hoặc `vegan >= veganFriendly`).

### Lỗi 2 — Quên tiêu chí phụ `id`

In [24]:
def wrong_no_tiebreak(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants
            if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=lambda r: -r[1])          # SAI: hoà rating thì giữ thứ tự đầu vào
    return [r[0] for r in kept]

print("sai :", wrong_no_tiebreak(rest, vf, mp, md_), "| đúng:", Solution().filterRestaurants(rest, vf, mp, md_))

sai : [4, 2, 3, 1, 5] | đúng: [4, 3, 2, 1, 5]


**Vì sao sai:** quán `2` và `3` cùng rating 8; sắp ổn định giữ thứ tự đầu vào (`2` trước `3`), nhưng đề yêu cầu id giảm dần (`3` trước `2`). **Sửa:** khoá `(-rating, -id)`.

### Lỗi 3 — Dùng `<` thay vì `<=`

In [25]:
def wrong_strict(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [(r[1], r[0]) for r in restaurants
            if r[2] >= veganFriendly and r[3] < maxPrice and r[4] < maxDistance]    # SAI
    kept.sort(reverse=True)
    return [i for _, i in kept]

rest, vf, mp, md_, want = examples[0]
print("sai :", wrong_strict(rest, vf, mp, md_), "| đúng:", Solution().filterRestaurants(rest, vf, mp, md_))

sai : [3, 5] | đúng: [3, 1, 5]


**Vì sao sai:** "max" là **giới hạn được phép** — quán `1` có `distance = 10 = maxDistance` vẫn hợp lệ. **Sửa:** `<=`.

### Lỗi 4 — Sắp hai lượt sai thứ tự (chính trước, phụ sau)

In [26]:
def wrong_two_pass_order(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants
            if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=lambda r: r[1], reverse=True)     # SAI: chính trước
    kept.sort(key=lambda r: r[0], reverse=True)     # lượt sau phá thứ tự rating
    return [r[0] for r in kept]

rest, vf, mp, md_, want = examples[1]
print("sai :", wrong_two_pass_order(rest, vf, mp, md_), "| đúng:", Solution().filterRestaurants(rest, vf, mp, md_))

sai : [5, 4, 3, 2, 1] | đúng: [4, 3, 2, 1, 5]


**Vì sao sai:** lượt sắp **cuối cùng** quyết định thứ tự chính. Sắp theo id ở lượt cuối ⇒ kết quả chỉ là id giảm dần. **Sửa:** sắp theo tiêu chí **phụ trước**, tiêu chí **chính sau**.

### Lỗi 5 — Trả về chỉ số mảng thay vì `id`

In [27]:
def wrong_index(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [(r[1], i) for i, r in enumerate(restaurants)        # SAI: dùng vị trí i
            if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(reverse=True)
    return [i for _, i in kept]

rest = [[10, 1, 0, 1, 1], [2, 3, 0, 1, 1]]
print("sai :", wrong_index(rest, 0, 5, 5), "| đúng:", Solution().filterRestaurants(rest, 0, 5, 5))

sai : [1, 0] | đúng: [2, 10]


**Vì sao sai:** gợi ý của đề nhắc: "id có thể không phải chỉ số trong mảng". Vị trí `i` vừa trả sai giá trị vừa phá tiêu chí phụ. **Sửa:** dùng `r[0]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "**lọc theo điều kiện rồi sắp theo nhiều tiêu chí**" ⇒ comprehension + `sort(key=tuple)`.
- Mọi tiêu chí cùng chiều giảm ⇒ `reverse=True` trên tuple; khác chiều ⇒ **đổi dấu** tiêu chí số, hoặc **sắp nhiều lượt** (phụ trước, chính sau).
- Đọc kỹ cờ boolean: "true ⇒ chỉ lấy…, false ⇒ lấy tất cả" ⇒ điều kiện `not flag or prop`.
- "max…/min…" trong đề thường là **bao gồm biên** (`<=`, `>=`).
- `id` ≠ chỉ số: luôn lấy từ dữ liệu.

## 15. Biến thể của bài

1. **Top-k quán** (Bài tập 5): chỉ cần `k` quán tốt nhất ⇒ `heapq.nlargest(k, ...)` `O(n log k)`.
2. **Nhiều truy vấn** với các bộ lọc khác nhau trên cùng danh sách: sắp sẵn một lần theo `(rating, id)` giảm, mỗi truy vấn chỉ lọc ⇒ `O(n)` mỗi truy vấn, không cần sắp lại.
3. Tiêu chí khác chiều: rating giảm dần, **giá tăng dần**, id giảm dần ⇒ khoá `(-rating, price, -id)`.
4. **LeetCode 1636 — Sort Array by Increasing Frequency**, **LeetCode 2418 — Sort the People:** cùng kỹ năng sắp theo khoá tuple.
5. **LeetCode 1366 — Rank Teams by Votes:** sắp theo vector đếm + tiêu chí phụ chữ cái.

In [28]:
def multi_query(restaurants, queries):
    """Biến thể 2: sắp một lần, mỗi truy vấn lọc O(n) giữ nguyên thứ tự đã sắp."""
    pre = sorted(restaurants, key=lambda r: (-r[1], -r[0]))
    return [[r[0] for r in pre if r[2] >= vf and r[3] <= mp and r[4] <= md_] for vf, mp, md_ in queries]

qs = [(1, 50, 10), (0, 50, 10), (0, 30, 3)]
print(multi_query(R, qs))
assert multi_query(R, qs) == [e[4] for e in examples]

def filter_price_tiebreak(restaurants, vf, mp, md_):
    """Biến thể 3: rating giảm, giá tăng, id giảm."""
    kept = [r for r in restaurants if r[2] >= vf and r[3] <= mp and r[4] <= md_]
    return [r[0] for r in sorted(kept, key=lambda r: (-r[1], r[3], -r[0]))]

print(filter_price_tiebreak([[1, 8, 1, 30, 1], [2, 8, 1, 20, 1], [3, 8, 1, 30, 1]], 0, 100, 100))

[[3, 1, 5], [4, 3, 2, 1, 5], [4, 5]]
[2, 3, 1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Selection sort tự viết | tìm quán tốt nhất còn lại mỗi bước | `O(n + k²)` | `O(k)` | hiểu bản chất so sánh, `k` nhỏ |
| 2. Key đổi dấu | `key=(-rating, -id)` | `O(n + k log k)` | `O(k)` | mặc định, rõ ràng |
| 3. Cặp + `reverse=True` | sắp tuple `(rating, id)` giảm | `O(n + k log k)` | `O(k)` | ngắn và nhanh nhất |
| 4. Hai lượt ổn định | id↓ rồi rating↓ | `O(n + k log k)` | `O(k)` | tiêu chí khác chiều, không đổi dấu được |
| 5. `cmp_to_key` | hàm so sánh tuỳ ý | `O(n + k log k)` | `O(k)` | quy tắc phức tạp, dịch từ Java/C++ |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Với mảng `R` của đề, `filterRestaurants(R, 1, 30, 10)` trả về gì?

### Bài 2 (tính tay)
`restaurants = [[5,7,0,20,3],[8,7,1,25,9],[2,9,1,40,2],[6,7,1,10,1]]`, `veganFriendly = 1, maxPrice = 30, maxDistance = 10`. Ghi bảng lọc và kết quả.

### Bài 3 (điền TODO)
Hoàn thiện `filter_todo` theo Cách 3.

### Bài 4 (tự viết lại)
Viết lại Cách 4 (hai lượt ổn định) cho quy tắc: **rating giảm dần, hoà thì id TĂNG dần**, không đổi dấu.

### Bài 5 (mini challenge)
Viết `top_k(restaurants, veganFriendly, maxPrice, maxDistance, k)` trả `k` id tốt nhất (đúng thứ tự) bằng `heapq` trong `O(n log k)`. Đối chiếu với `Solution(...)[:k]`.

In [29]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def filter_todo(restaurants, veganFriendly, maxPrice, maxDistance):
    pairs = []
    # TODO: duyệt, giữ (rating, id) nếu thoả 3 điều kiện
    # TODO: sắp giảm dần
    # TODO: trả danh sách id
    pass

def filter_id_asc_todo(restaurants, veganFriendly, maxPrice, maxDistance):
    # TODO: hai lượt ổn định
    pass

def top_k_todo(restaurants, veganFriendly, maxPrice, maxDistance, k):
    # TODO: heapq.nlargest hoặc min-heap kích thước k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Lọc chay: còn quán 1, 3, 5. (2) Giá ≤ 30: quán 1 (40) bị loại. (3) Còn 3 (8), 5 (1) ⇒ `[3,5]`.

**Bài 2:** (1) Quán 5 không chay ⇒ loại; quán 2 giá 40 > 30 ⇒ loại. (2) Còn 8 (rating 7) và 6 (rating 7). (3) Cùng rating ⇒ id giảm ⇒ `[8,6]`.

**Bài 3:** (1) `for id_, rating, vegan, price, dist in restaurants`. (2) `pairs.sort(reverse=True)`. (3) `[i for _, i in pairs]`.

**Bài 4:** (1) Lượt 1: `sort(key=id)` tăng dần. (2) Lượt 2: `sort(key=rating, reverse=True)`. (3) Kiểm chứng bằng `sorted(key=(-rating, id))`.

**Bài 5:** (1) `heapq.nlargest(k, iterable)` trả `k` phần tử lớn nhất **đã sắp giảm**. (2) Iterable là các cặp `(rating, id)` sau lọc. (3) Hoặc tự giữ min-heap kích thước `k`, cuối cùng sắp giảm.

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
b1 = Solution().filterRestaurants(R, 1, 30, 10)
R2 = [[5, 7, 0, 20, 3], [8, 7, 1, 25, 9], [2, 9, 1, 40, 2], [6, 7, 1, 10, 1]]
b2 = Solution().filterRestaurants(R2, 1, 30, 10)
print("Bài 1:", b1, "| Bài 2:", b2)
assert b1 == [3, 5] and b2 == [8, 6]

# Bài 3
def filter_sol(restaurants, veganFriendly, maxPrice, maxDistance):
    pairs = []
    for id_, rating, vegan, price, dist in restaurants:
        if (veganFriendly == 0 or vegan == 1) and price <= maxPrice and dist <= maxDistance:
            pairs.append((rating, id_))
    pairs.sort(reverse=True)
    return [i for _, i in pairs]

for c in cases:
    assert filter_sol(*c[:4]) == c[4]

# Bài 4
def filter_id_asc(restaurants, veganFriendly, maxPrice, maxDistance):
    kept = [r for r in restaurants if r[2] >= veganFriendly and r[3] <= maxPrice and r[4] <= maxDistance]
    kept.sort(key=lambda r: r[0])
    kept.sort(key=lambda r: r[1], reverse=True)
    return [r[0] for r in kept]

rng = random.Random(4)
for _ in range(300):
    args = random_case(rng, rng.randint(1, 25))
    kept = [r for r in args[0] if r[2] >= args[1] and r[3] <= args[2] and r[4] <= args[3]]
    assert filter_id_asc(*args) == [r[0] for r in sorted(kept, key=lambda r: (-r[1], r[0]))]
print("Bài 1–4: qua")

Bài 1: [3, 5] | Bài 2: [8, 6]
Bài 1–4: qua


In [31]:
# Bài 5
def top_k(restaurants, veganFriendly, maxPrice, maxDistance, k):
    gen = ((rating, id_) for id_, rating, vegan, price, dist in restaurants
           if vegan >= veganFriendly and price <= maxPrice and dist <= maxDistance)
    return [id_ for _, id_ in heapq.nlargest(k, gen)]

def top_k_manual(restaurants, veganFriendly, maxPrice, maxDistance, k):
    heap = []                                  # min-heap giữ k phần tử lớn nhất
    for id_, rating, vegan, price, dist in restaurants:
        if vegan >= veganFriendly and price <= maxPrice and dist <= maxDistance:
            heapq.heappush(heap, (rating, id_))
            if len(heap) > k:
                heapq.heappop(heap)
    return [id_ for _, id_ in sorted(heap, reverse=True)]

print("Bài 5:", top_k(R, 0, 50, 10, 2), top_k_manual(R, 0, 50, 10, 2))
rng = random.Random(5)
for _ in range(500):
    args = random_case(rng, rng.randint(1, 30))
    k = rng.randint(1, 10)
    full = Solution().filterRestaurants(*args)
    assert top_k(*args, k) == full[:k] == top_k_manual(*args, k)
print("Bài 5: 500 bộ ngẫu nhiên khớp lời giải đầy đủ")

Bài 5: [4, 3] [4, 3]
Bài 5: 500 bộ ngẫu nhiên khớp lời giải đầy đủ


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template lọc + sắp nhiều tiêu chí:**

```python
kept = [(rating, id_) for id_, rating, vegan, price, dist in restaurants
        if (veganFriendly == 0 or vegan == 1)
        and price <= maxPrice and dist <= maxDistance]
kept.sort(reverse=True)           # (rating giảm, id giảm)
return [id_ for _, id_ in kept]
# khác chiều: key=lambda r: (-r[1], r[0])  hoặc sắp nhiều lượt: phụ trước, chính sau
```

**Checklist trước khi submit:**
- [ ] `veganFriendly = 0` ⇒ **không lọc** chay.
- [ ] Giá và khoảng cách dùng `<=`.
- [ ] Tiêu chí phụ `id` giảm dần khi cùng rating.
- [ ] Trả về **id**, không phải chỉ số hay cả hàng.
- [ ] Nếu sắp nhiều lượt: phụ trước, chính sau.
- [ ] Đúng chữ ký: `filterRestaurants(self, restaurants, veganFriendly, maxPrice, maxDistance) -> list[int]`.